# Project 01 — Customer Churn Prediction Using a 12-Model Machine Learning Benchmarking Framework

## Executive Summary

Customer churn is one of the most commercially important problems for subscription-based businesses. In sectors such as telecommunications, banking, insurance, SaaS, and financial services, retaining an existing customer is often significantly more cost-effective than acquiring a new one.

In this project, I developed an end-to-end machine learning pipeline to predict which customers are most likely to churn. The aim was not simply to build a high-performing model, but to create a commercially interpretable solution that could support real-world customer retention decisions.

  version includes:

- Leakage-safe train/test workflow
- One-hot encoding instead of inappropriate ordinal label encoding
- Feature selection after the train/test split
- 5-fold cross-validation with mean and standard deviation of F1 Score
- 12-model benchmarking framework
- SHAP explainability
- Business-focused recommendations
- Deployment considerations

The overall objective is to demonstrate technical competence, commercial awareness, and production-oriented thinking.

In [ ]:
# ============================================================
# CUSTOMER CHURN PREDICTION — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

print("All libraries loaded successfully.")

# 1. Data Loading and Initial Exploration

The first stage of the project is to load the dataset and understand its basic structure.

Before modelling, I check:

- Dataset shape
- Overall churn rate
- Feature structure
- Basic data quality

This is important because churn datasets often contain class imbalance, missing values, and mixed data types that must be handled carefully before training machine learning models.

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv('telco_churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"Customer churn rate: {(df['Churn'] == 'Yes').mean():.2%}")

df.head()

# 2. Exploratory Data Analysis

Exploratory Data Analysis helps identify early commercial patterns before building predictive models.

In this section, I examine:

- Overall churn distribution
- Churn rate by contract type
- Monthly charge differences between retained and churned customers

This step is valuable because it connects technical findings back to commercial behaviour. For example, if month-to-month customers churn more frequently, this insight can directly inform retention campaigns and contract conversion strategies.

In [ ]:
# ============================================================
# EXPLORATORY DATA ANALYSIS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn distribution
df['Churn'].value_counts().plot(
    kind='bar',
    ax=axes[0],
    color=['#0F1C2E', '#C9A96E']
)

axes[0].set_title('Customer Churn Distribution', fontweight='bold')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)
axes[0].set_ylabel('Customer Count')

# Churn by contract type
churn_contract = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
)

churn_contract.plot(
    kind='bar',
    ax=axes[1],
    color='#C9A96E',
    edgecolor='#0F1C2E'
)

axes[1].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[1].set_ylabel('Churn Rate')
axes[1].set_xlabel('Contract Type')

# Monthly charges by churn outcome
df[df['Churn'] == 'No']['MonthlyCharges'].plot(
    kind='hist',
    bins=40,
    ax=axes[2],
    alpha=0.6,
    color='#0F1C2E',
    label='Retained'
)

df[df['Churn'] == 'Yes']['MonthlyCharges'].plot(
    kind='hist',
    bins=40,
    ax=axes[2],
    alpha=0.6,
    color='#C9A96E',
    label='Churned'
)

axes[2].set_title('Monthly Charges by Churn Outcome', fontweight='bold')
axes[2].set_xlabel('Monthly Charges')
axes[2].legend()

plt.suptitle(
    'Exploratory Analysis of Customer Churn Behaviour',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

# 3. Data Cleaning, Feature Engineering and One-Hot Encoding

This section prepares the data for machine learning.

Key improvements in this  version:

- `LabelEncoder` has been removed for categorical predictors
- Nominal categorical variables are encoded using `pd.get_dummies()`
- Additional commercial features are engineered before encoding
- The target variable is separated before one-hot encoding

This matters because many categorical variables in the dataset are nominal rather than ordinal. For example, payment method and contract type do not have a natural numeric order. Using one-hot encoding avoids accidentally implying false ranking relationships between categories.

A key churn modelling risk is data leakage. `TotalCharges` can be misleading because customers with low tenure naturally have lower accumulated charges. To reduce this risk, I engineer tenure-adjusted and value-based features such as `AvgMonthlySpend` and `CostPerService`.

In [ ]:
# ============================================================
# DATA CLEANING, FEATURE ENGINEERING AND ONE-HOT ENCODING
# ============================================================

df_model = df.copy()

# Remove unique customer identifier
df_model.drop('customerID', axis=1, inplace=True)

# Convert TotalCharges to numeric
df_model['TotalCharges'] = pd.to_numeric(
    df_model['TotalCharges'],
    errors='coerce'
)

# Fill missing TotalCharges values
df_model['TotalCharges'] = df_model['TotalCharges'].fillna(
    df_model['TotalCharges'].median()
)

# Convert target to binary
df_model['Churn'] = (df_model['Churn'] == 'Yes').astype(int)

# Feature engineering before encoding
df_model['AvgMonthlySpend'] = df_model['TotalCharges'] / (df_model['tenure'] + 1)

service_cols_raw = [
    'PhoneService',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

df_model['ServiceCount'] = df_model[service_cols_raw].apply(
    lambda row: sum(row.astype(str).isin(['Yes', 'Fiber optic', 'DSL'])),
    axis=1
)

df_model['CostPerService'] = df_model['MonthlyCharges'] / (df_model['ServiceCount'] + 1)

# Split target before one-hot encoding
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# One-hot encode nominal categorical features
X = pd.get_dummies(X, drop_first=True)

print(f"Total samples: {X.shape[0]}")
print(f"Total features after one-hot encoding: {X.shape[1]}")
print("Target distribution:")
print(y.value_counts(normalize=True))

# 4. Train/Test Split

The dataset is split before feature selection to prevent data leakage.

This is important because if feature selection is performed on the full dataset before splitting, information from the test set can influence which features are selected. That would make test performance look better than it really is.

Using a leakage-safe split ensures the final evaluation better reflects how the model would behave on genuinely unseen customers.

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# 5. Leakage-Safe Chi-Square Feature Selection

Chi-square feature selection is now performed after the train/test split.

This prevents the test set from influencing feature importance calculations.

Chi-square requires non-negative values, so the training data is shifted into a non-negative range. The test data is shifted using the training-set minimums only, preserving leakage-safe preprocessing.

In [ ]:
# ============================================================
# LEAKAGE-SAFE CHI-SQUARE FEATURE SELECTION
# ============================================================

imputer_fs = SimpleImputer(strategy='median')

X_train_imputed = pd.DataFrame(
    imputer_fs.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer_fs.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Chi-square requires non-negative values
X_train_pos = X_train_imputed - X_train_imputed.min()
X_test_pos = X_test_imputed - X_train_imputed.min()

selector = SelectKBest(score_func=chi2, k='all')
selector.fit(X_train_pos, y_train)

chi2_scores = pd.Series(
    selector.scores_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 10 features by chi-square importance:")
print(chi2_scores.head(10))

plt.figure(figsize=(12, 5))
chi2_scores.head(15).plot(
    kind='bar',
    color='#C9A96E',
    edgecolor='#0F1C2E'
)

plt.title('Leakage-Safe Chi-Square Feature Importance', fontweight='bold')
plt.ylabel('Chi-Square Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 6. Final Preprocessing for Model Training

The final preprocessing stage applies imputation and scaling.

To avoid leakage:

- The imputer is fitted only on the training data
- The scaler is fitted only on the training data
- The test data is transformed using training-set parameters only

This mirrors how a real deployed model would process new unseen customers.

In [ ]:
# ============================================================
# FINAL PREPROCESSING FOR MODEL TRAINING
# ============================================================

imputer = SimpleImputer(strategy='median')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()

X_tr = scaler.fit_transform(X_train_imputed)
X_te = scaler.transform(X_test_imputed)

print("NaNs in training data:", np.isnan(X_tr).sum())
print("NaNs in testing data:", np.isnan(X_te).sum())

# 7. Machine Learning Benchmarking with Cross-Validation

This project benchmarks 12 classification models.

A key upgrade in this version is the inclusion of 5-fold stratified cross-validation. This provides a more robust assessment of model performance because it reports both:

- Mean F1 Score
- Standard deviation of F1 Score

The standard deviation matters because it indicates model stability. A model with a slightly lower mean score but lower variance may be preferable in a real business environment.

In [ ]:
# ============================================================
# TRAIN 12 MODELS WITH CROSS-VALIDATION
# ============================================================

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'MLP Neural Net': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric='logloss'
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}
trained_pipelines = {}

for name, model in models.items():

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1'
    )

    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:, 1]

    test_f1 = f1_score(y_test, pred)
    test_auc = roc_auc_score(y_test, prob)

    results[name] = {
        'CV F1 Mean': round(cv_scores.mean(), 4),
        'CV F1 Std': round(cv_scores.std(), 4),
        'Test F1': round(test_f1, 4),
        'Test AUC': round(test_auc, 4)
    }

    trained_pipelines[name] = pipeline

    print(
        f"{name:<25} "
        f"CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | "
        f"Test F1: {test_f1:.4f} | "
        f"AUC: {test_auc:.4f}"
    )

# 8. Model Comparison

The models are compared using:

- Cross-validated F1 mean
- Cross-validated F1 standard deviation
- Test F1 Score
- Test AUC-ROC

F1 Score is prioritised because churn prediction is an imbalanced classification problem. Accuracy alone may be misleading if the model performs well on retained customers but fails to detect genuine churners.

In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results).T.sort_values('Test F1', ascending=False)

display(results_df)

fig, ax = plt.subplots(figsize=(12, 6))

results_df[['CV F1 Mean', 'Test F1', 'Test AUC']].plot(
    kind='bar',
    ax=ax,
    color=['#0F1C2E', '#C9A96E', '#7A869A'],
    edgecolor='white'
)

ax.set_title(
    'Model Comparison: Cross-Validated F1, Test F1 and AUC-ROC',
    fontsize=14,
    fontweight='bold'
)

ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

best_model_name = results_df['Test F1'].idxmax()

print(f"Best model by Test F1 Score: {best_model_name}")
print(results_df.loc[best_model_name])

# 9. Final Model Selection

The final model is selected using Test F1 Score after cross-validation benchmarking.

However, in a real organisation, the final deployment model would not be selected solely on performance. Practical considerations include:

- Interpretability
- Inference speed
- Monitoring complexity
- Business stakeholder trust
- Ease of deployment
- Maintenance requirements

This is why both performance and explainability are included in the project.

In [ ]:
# ============================================================
# FINAL MODEL SELECTION
# ============================================================

best_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', models[best_model_name])
])

best_model.fit(X_train, y_train)

print(f"Selected final model: {best_model_name}")
print("Model selected using test F1 Score after cross-validation benchmarking.")

# 10. Confusion Matrix and ROC Curve

The confusion matrix helps translate model performance into business terms.

It shows:

- True retained customers
- Incorrectly flagged retained customers
- Missed churners
- Correctly identified churners

This distinction matters because different businesses may optimise for different outcomes.

If retention budget is limited, precision may matter more.  
If preventing churn is the priority, recall may matter more.

In [ ]:
# ============================================================
# CONFUSION MATRIX AND ROC CURVE
# ============================================================

pred = best_model.predict(X_test)
prob = best_model.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, pred)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0],
    xticklabels=['Retained', 'Churned'],
    yticklabels=['Retained', 'Churned']
)

axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

fpr, tpr, thresholds = roc_curve(y_test, prob)
auc_score = roc_auc_score(y_test, prob)

axes[1].plot(
    fpr,
    tpr,
    color='#C9A96E',
    lw=2,
    label=f'AUC = {auc_score:.4f}'
)

axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print(classification_report(
    y_test,
    pred,
    target_names=['Retained', 'Churned']
))

# 11. SHAP Explainability

SHAP is used to explain model predictions.

In a real business environment, stakeholders do not only want to know which customers are likely to churn. They also want to understand why.

SHAP helps answer:

- Which features contributed most to churn risk?
- Did a feature increase or decrease the prediction?
- Are the model findings commercially intuitive?
- Can retention teams trust and act on the output?

If the best-performing model is not tree-based, a Random Forest model is used as an explainability benchmark because SHAP is particularly effective with tree-based algorithms.

In [ ]:
# ============================================================
# SHAP EXPLAINABILITY
# ============================================================

tree_based_models = [
    'Decision Tree',
    'Random Forest',
    'Gradient Boosting',
    'Extra Trees',
    'XGBoost'
]

if best_model_name in tree_based_models:
    shap_pipeline = best_model
    shap_model_name = best_model_name
else:
    shap_pipeline = trained_pipelines['Random Forest']
    shap_model_name = 'Random Forest'

print(f"SHAP explainability model: {shap_model_name}")

# Transform data through preprocessing steps
X_test_processed = shap_pipeline.named_steps['imputer'].transform(X_test)
X_test_processed = shap_pipeline.named_steps['scaler'].transform(X_test_processed)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=X.columns
)

model_for_shap = shap_pipeline.named_steps['model']

explainer = shap.TreeExplainer(model_for_shap)
shap_values = explainer.shap_values(X_test_processed_df.iloc[:200])

if isinstance(shap_values, list):
    shap_values_to_plot = shap_values[1]
else:
    shap_values_to_plot = shap_values

shap.summary_plot(
    shap_values_to_plot,
    X_test_processed_df.iloc[:200],
    feature_names=list(X.columns),
    plot_type='bar'
)

# 12. Business Interpretation

The model results highlight several commercially meaningful churn drivers.

## Key Finding 1: Month-to-Month Contracts

Customers on month-to-month contracts are more likely to churn because they have lower switching friction.

## Key Finding 2: High Monthly Charges

Customers with higher monthly charges may be more price-sensitive or may feel that they are not receiving sufficient value for money.

## Key Finding 3: Low Tenure

Customers in the earlier stages of their relationship are more vulnerable to churn, suggesting that onboarding and early customer experience are critical.

## Key Finding 4: Fibre Optic Customers

Fibre optic customers may show elevated churn despite often generating higher revenue. This could indicate pricing dissatisfaction, service reliability concerns, or stronger competitor alternatives.

## Key Finding 5: Cost Per Service

Customers paying more for fewer services may have a weaker perceived value relationship with the provider.

In [ ]:
# ============================================================
# BUSINESS RECOMMENDATIONS
# ============================================================

print("""
==============================
BUSINESS RECOMMENDATIONS
==============================

1. TARGET MONTH-TO-MONTH CUSTOMERS
Customers on flexible contracts are easier to lose because they have lower switching friction.

Recommended action:
Offer loyalty incentives to move high-risk month-to-month customers onto longer-term contracts.

------------------------------------------------------------

2. PROTECT HIGH-VALUE CUSTOMERS
Customers with high monthly charges may be more price-sensitive.

Recommended action:
Introduce personalised loyalty pricing, bundle optimisation, or targeted retention discounts.

------------------------------------------------------------

3. IMPROVE EARLY CUSTOMER LIFECYCLE RETENTION
Low-tenure customers are more likely to churn.

Recommended action:
Create proactive onboarding campaigns at months 1, 3, 6, and 12.

------------------------------------------------------------

4. INVESTIGATE FIBRE OPTIC CHURN
Fibre optic customers may generate higher revenue but also show elevated churn risk.

Recommended action:
Analyse whether churn is driven by price, service reliability, customer support, or competitor offers.

------------------------------------------------------------

5. USE RISK-BASED RETENTION TARGETING
Not every customer should receive the same retention offer.

Recommended action:
Use predicted churn probabilities to prioritise high-risk, high-value customers where intervention is most likely to protect revenue.

==============================
""")

# 13. Deployment Considerations

A production-ready churn model could be deployed in several ways.

## Weekly Batch Scoring

A scheduled process could score all active customers weekly and send high-risk customers to CRM or retention teams.

Suitable for:

- Marketing campaigns
- Retention dashboards
- Account management workflows

## Real-Time API Scoring

A real-time API could generate churn-risk scores during customer interactions.

Suitable for:

- Call centre systems
- Customer support workflows
- Personalised offer engines

## Production Requirements

A production version would also require:

- Model monitoring
- Data drift detection
- Retraining schedules
- Threshold optimisation
- Logging and auditability
- Fairness checks
- CRM integration

# 14. Final Conclusion

This project demonstrates an end-to-end customer churn prediction workflow using machine learning.

The  improves the project by addressing four important professional standards:

1. Feature selection is performed after the train/test split to prevent leakage
2. Nominal categories are encoded properly using one-hot encoding
3. Cross-validation is included to assess model stability
4. The README is structured for recruiter readability by placing findings and recommendations first

The project demonstrates capability across:

- Data analysis
- Feature engineering
- Leakage-safe machine learning
- Classification modelling
- Model evaluation
- Cross-validation
- Explainable AI
- Commercial recommendation development
- Production thinking

From a business perspective, the model could help retention teams identify high-risk customers earlier and target interventions more efficiently.

From a technical perspective, the project demonstrates awareness of class imbalance, data leakage, encoding strategy, model selection trade-offs, and explainability.